In [2]:
:dep sha2 = "0.10"
:dep hex = "0.4"
:dep aes = "0.8"
:dep cbc = "0.1"
:dep block-modes = "0.9"

use sha2::{Sha256, Digest};
use hex;
use aes::Aes128;
use cbc::Decryptor;
use cbc::cipher::{BlockDecryptMut, KeyIvInit};

In [3]:
fn solve_device_id() -> Option<String> {
    let target_hash = "356280a58d3c437a45268a0b226d8cccad7b5dd28f5d1b37abf1873cc426a8a5";
    let prefix = "99999991";
    
    // 残り7桁（IMEIは通常15桁）を総当たり
    for i in 0..10_000_000 {
        let device_id = format!("{}{:07}", prefix, i);
        let mut hasher = Sha256::new();
        hasher.update(device_id.as_bytes());
        let result = hasher.finalize();
        
        if hex::encode(result) == target_hash {
            return Some(device_id);
        }
    }
    None
}

if let Some(id) = solve_device_id() {
    println!("Found Device ID: {}", id);
} else {
    println!("Device ID not found.");
}

Found Device ID: 999999913371337


()

In [6]:
{
    // 前のステップで見つかったID
    let device_id = "999999913371337";
    let key_str = format!("!{}", device_id);
    
    // &strではなくVec<u8>に変換して所有権を持たせる
    let key = key_str.as_bytes().to_vec(); 
    let iv = b"kLwC29iMc4nRMuE5";

    // ファイル読み込み
    let mut data = std::fs::read("jewel_c.png").expect("Failed to read encrypted file");

    type Aes128CbcDec = cbc::Decryptor<aes::Aes128>;

    let cipher = Aes128CbcDec::new(key.as_slice().into(), iv.into());

    // 復号化（全体をブロックで囲むことで参照のライフタイム問題を回避）
    let decrypted_data = cipher.decrypt_padded_mut::<cbc::cipher::block_padding::Pkcs7>(&mut data)
        .expect("Decryption failed");

    std::fs::write("decrypted_jewel.png", decrypted_data).expect("Failed to save image");
    
    println!("Success! Check decrypted_jewel.png");
}

Success! Check decrypted_jewel.png


()